# Data Preprocessing Pipeline
## Stock News Dataset - Complete Preprocessing

This notebook contains the complete preprocessing pipeline including:
- Date feature extraction
- Missing value handling
- Text combination and cleaning
- Text feature engineering
- TF-IDF vectorization
- Feature scaling
- Train-test split

### IMPORTS

In [30]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.feature_extraction.text import TfidfVectorizer
import pickle
import warnings
warnings.filterwarnings("ignore")

print("✅ All libraries imported successfully!")

✅ All libraries imported successfully!


### LOADING DATA

In [31]:
df = pd.read_csv("../data/raw/Combined_News_DJIA.csv")
df.head()

,Date,Label,Top1,Top2,Top3,Top4,Top5,Top6,Top7,Top8,...,Top16,Top17,Top18,Top19,Top20,Top21,Top22,Top23,Top24,Top25
0,2008-08-08,0,"b""Georgia 'downs two Russian warplanes' as cou...",b'BREAKING: Musharraf to be impeached.',b'Russia Today: Columns of troops roll into So...,b'Russian tanks are moving towards the capital...,"b""Afghan children raped with 'impunity,' U.N. ...",b'150 Russian tanks have entered South Ossetia...,"b""Breaking: Georgia invades South Ossetia, Rus...","b""The 'enemy combatent' trials are nothing but...",...,b'Georgia Invades South Ossetia - if Russia ge...,b'Al-Qaeda Faces Islamist Backlash',"b'Condoleezza Rice: ""The US would not act to p...",b'This is a busy day: The European Union has ...,"b""Georgia will withdraw 1,000 soldiers from Ir...",b'Why the Pentagon Thinks Attacking Iran is a ...,b'Caucasus in crisis: Georgia invades South Os...,b'Indian shoe manufactory - And again in a se...,b'Visitors Suffering from Mental Illnesses Ban...,"b""No Help for Mexico's Kidnapping Surge"""
1,2008-08-11,1,b'Why wont America and Nato help us? If they w...,b'Bush puts foot down on Georgian conflict',"b""Jewish Georgian minister: Thanks to Israeli ...",b'Georgian army flees in disarray as Russians ...,"b""Olympic opening ceremony fireworks 'faked'""",b'What were the Mossad with fraudulent New Zea...,b'Russia angered by Israeli military sale to G...,b'An American citizen living in S.Ossetia blam...,...,b'Israel and the US behind the Georgian aggres...,"b'""Do not believe TV, neither Russian nor Geor...",b'Riots are still going on in Montreal (Canada...,b'China to overtake US as largest manufacturer',b'War in South Ossetia [PICS]',b'Israeli Physicians Group Condemns State Tort...,b' Russia has just beaten the United States ov...,b'Perhaps *the* question about the Georgia - R...,b'Russia is so much better at war',"b""So this is what it's come to: trading sex fo..."
2,2008-08-12,0,b'Remember that adorable 9-year-old who sang a...,"b""Russia 'ends Georgia operation'""","b'""If we had no sexual harassment we would hav...","b""Al-Qa'eda is losing support in Iraq because ...",b'Ceasefire in Georgia: Putin Outmaneuvers the...,b'Why Microsoft and Intel tried to kill the XO...,b'Stratfor: The Russo-Georgian War and the Bal...,"b""I'm Trying to Get a Sense of This Whole Geor...",...,b'U.S. troops still in Georgia (did you know t...,b'Why Russias response to Georgia was right',"b'Gorbachev accuses U.S. of making a ""serious ...","b'Russia, Georgia, and NATO: Cold War Two'",b'Remember that adorable 62-year-old who led y...,b'War in Georgia: The Israeli connection',b'All signs point to the US encouraging Georgi...,b'Christopher King argues that the US and NATO...,b'America: The New Mexico?',"b""BBC NEWS | Asia-Pacific | Extinction 'by man..."
3,2008-08-13,0,b' U.S. refuses Israel weapons to attack Iran:...,"b""When the president ordered to attack Tskhinv...",b' Israel clears troops who killed Reuters cam...,b'Britain\'s policy of being tough on drugs is...,b'Body of 14 year old found in trunk; Latest (...,b'China has moved 10 *million* quake survivors...,"b""Bush announces Operation Get All Up In Russi...",b'Russian forces sink Georgian ships ',...,b'Elephants extinct by 2020?',b'US humanitarian missions soon in Georgia - i...,"b""Georgia's DDOS came from US sources""","b'Russian convoy heads into Georgia, violating...",b'Israeli defence minister: US against strike ...,b'Gorbachev: We Had No Choice',b'Witness: Russian forces head towards Tbilisi...,b' Quarter of Russians blame U.S. for conflict...,b'Georgian president says US military will ta...,b'2006: Nobel laureate Aleksander Solzhenitsyn...
4,2008-08-14,1,b'All the experts admit that we should legalis...,b'War in South Osetia - 89 pictures made by a ...,b'Swedish wrestler Ara Abrahamian throws away ...,b'Russia exaggerated the death toll in South O...,b'Missile That Killed 9 Inside Pakistan May Ha...,"b""Rushdie Condemns Random House's Refusal to P...",b'Poland and US agree 

### DATE FEATURE EXTRACTION
Converting date column to datetime and extracting temporal features.

In [32]:
df['Date'] = pd.to_datetime(df['Date'], errors='coerce')
df['Year'] = df['Date'].dt.year
df['Month'] = df['Date'].dt.month
df['Day'] = df['Date'].dt.day
df['DayOfWeek'] = df['Date'].dt.dayofweek
df['Quarter'] = df['Date'].dt.quarter

print("✅ Date features extracted:")
print(df[['Date', 'Year', 'Month', 'Day', 'DayOfWeek', 'Quarter']].head())

✅ Date features extracted:
        Date  Year  Month  Day  DayOfWeek  Quarter
0 2008-08-08  2008      8    8          4        3
1 2008-08-11  2008      8   11          0        3
2 2008-08-12  2008      8   12          1        3
3 2008-08-13  2008      8   13          2        3
4 2008-08-14  2008      8   14          3        3


### Missing Values Handling
Filling missing headline columns with empty strings.

In [33]:
print("Missing values before:")
print(df.isnull().sum()[df.isnull().sum() > 0])

# Fill missing headlines
headline_cols = [f'Top{i}' for i in range(1, 26)]
for col in headline_cols:
    if col in df.columns:
        df[col].fillna('', inplace=True)

print("\n✅ Missing values handled!")
print("Missing values after:")
print(df.isnull().sum().sum())


Missing values before:
Top23    1
Top24    3
Top25    3
dtype: int64

✅ Missing values handled!
Missing values after:
0


### Combining News Headlines
Merging all Top1-Top25 columns into single News column.

In [34]:
df['News'] = df[headline_cols].apply(lambda x: ' '.join(x.astype(str)), axis=1)

# Clean text
df['News'] = df['News'].str.replace("b'", "", regex=False)
df['News'] = df['News'].str.replace('b"', "", regex=False)
df['News'] = df['News'].str.replace("'", "", regex=False)

print("✅ News column created!")
print(f"\nSample (first 150 chars):")
print(df['News'].iloc[0][:150] + "...")
print(f"\nAvg length: {df['News'].apply(len).mean():.0f} characters")

✅ News column created!

Sample (first 150 chars):
Georgia downs two Russian warplanes as countries move to brink of war" BREAKING: Musharraf to be impeached. Russia Today: Columns of troops roll into ...

Avg length: 2752 characters


### Text Feature Engineering
Creating statistical features from the news text:
- Text length
- Word count
- Average word length
- Positive/negative keyword counts
- Sentiment score

In [35]:
# Basic text statistics
df['Text_Length'] = df['News'].apply(len)
df['Word_Count'] = df['News'].apply(lambda x: len(x.split()))
df['Avg_Word_Length'] = df['News'].apply(
    lambda x: np.mean([len(word) for word in x.split()]) if len(x.split()) > 0 else 0
)

# Sentiment keywords
positive_keywords = ['gain', 'rise', 'up', 'high', 'increase', 'profit', 'growth', 'win', 'success']
negative_keywords = ['loss', 'fall', 'down', 'low', 'decrease', 'crisis', 'crash', 'fail', 'drop']

def count_keywords(text, keywords):
    text_lower = text.lower()
    return sum(text_lower.count(keyword) for keyword in keywords)

df['Positive_Keywords'] = df['News'].apply(lambda x: count_keywords(x, positive_keywords))
df['Negative_Keywords'] = df['News'].apply(lambda x: count_keywords(x, negative_keywords))
df['Sentiment_Score'] = df['Positive_Keywords'] - df['Negative_Keywords']

print("✅ Text features created!")
print("\nText feature statistics:")
text_features = ['Text_Length', 'Word_Count', 'Avg_Word_Length', 
                 'Positive_Keywords', 'Negative_Keywords', 'Sentiment_Score']
print(df[text_features].describe())

✅ Text features created!

Text feature statistics:
       Text_Length   Word_Count  Avg_Word_Length  Positive_Keywords  \
count  1989.000000  1989.000000      1989.000000        1989.000000   
mean   2752.443439   441.979387         5.225453           5.936149   
std     445.032306    72.570635         0.152950           2.793432   
min    1393.000000   219.000000         4.699700           0.000000   
25%    2469.000000   395.000000         5.119792           4.000000   
50%    2768.000000   443.000000         5.218289           6.000000   
75%    3057.000000   491.000000         5.330846           8.000000   
max    4340.000000   717.000000         5.753049          22.000000   

       Negative_Keywords  Sentiment_Score  
count        1989.000000      1989.000000  
mean            2.629965         3.306184  
std             1.750202         3.159774  
min             0.000000        -7.000000  
25%             1.000000         1.000000  
50%             2.000000         3.000000  
7

### TF-IDF Vectorization
Converting text to numerical features using TF-IDF.
This is a CRITICAL step for machine learning!

In [36]:
print("⏳ Vectorizing text using TF-IDF...")

# Initialize TF-IDF
tfidf = TfidfVectorizer(max_features=2000, ngram_range=(1, 2), stop_words='english')
# Fit and transform
X_tfidf = tfidf.fit_transform(df['News'])

print(f"✅ TF-IDF vectorization complete!")
print(f"   Shape: {X_tfidf.shape}")
print(f"   Features: {X_tfidf.shape[1]}")

# Convert to DataFrame
tfidf_df = pd.DataFrame(
    X_tfidf.toarray(),
    columns=[f'tfidf_{i}' for i in range(X_tfidf.shape[1])]
)

print("\nTop 10 TF-IDF features:")
print(tfidf.get_feature_names_out()[:10])

⏳ Vectorizing text using TF-IDF...
✅ TF-IDF vectorization complete!
   Shape: (1989, 2000)
   Features: 2000

Top 10 TF-IDF features:
['000' '000 people' '10' '10 000' '10 years' '100' '100 000' '11' '12'
 '13']


### Combining All Features
Merging date features, text features, and TF-IDF features.

In [37]:
# Define feature groups
date_features = ['Year', 'Month', 'Day', 'DayOfWeek', 'Quarter']
text_stat_features = ['Text_Length', 'Word_Count', 'Avg_Word_Length', 
                      'Positive_Keywords', 'Negative_Keywords', 'Sentiment_Score']

# Combine all features
feature_df = df[date_features + text_stat_features].copy()
feature_df = pd.concat([feature_df.reset_index(drop=True), tfidf_df], axis=1)

print(f"✅ All features combined!")
print(f"\nTotal features: {feature_df.shape[1]}")
print(f"  - Date features: {len(date_features)}")
print(f"  - Text stat features: {len(text_stat_features)}")
print(f"  - TF-IDF features: {X_tfidf.shape[1]}")

print("\nFeature DataFrame shape:", feature_df.shape)
print("\nFirst few features:")
print(feature_df.head())


✅ All features combined!

Total features: 2011
  - Date features: 5
  - Text stat features: 6
  - TF-IDF features: 2000

Feature DataFrame shape: (1989, 2011)

First few features:
   Year  Month  Day  DayOfWeek  Quarter  Text_Length  Word_Count  \
0  2008      8    8          4        3         2288         374   
1  2008      8   11          0        3         1590         270   
2  2008      8   12          1        3         2084         349   
3  2008      8   13          2        3         1975         310   
4  2008      8   14          3        3         1757         275   

   Avg_Word_Length  Positive_Keywords  Negative_Keywords  ...  tfidf_1990  \
0         5.112299                  3                  3  ...         0.0   
1         4.881481                  3                  1  ...         0.0   
2         4.959885                  3                  2  ...         0.0   
3         5.358065                  4                  0  ...         0.0   
4         5.378182        

### Preparing Features and Target
Separating features (X) and target variable (y).

In [38]:
X = feature_df.values
y = df['Label'].values

print(f"✅ Features (X) shape: {X.shape}")
print(f"✅ Target (y) shape: {y.shape}")
print(f"\nTarget distribution:")
print(f"  Label 0: {np.sum(y == 0)}")
print(f"  Label 1: {np.sum(y == 1)}")

✅ Features (X) shape: (1989, 2011)
✅ Target (y) shape: (1989,)

Target distribution:
  Label 0: 924
  Label 1: 1065


### Train-Test Split
Splitting data into 80% training and 20% testing sets.

In [39]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, 
    test_size=0.2, 
    random_state=42, 
    stratify=y
)

print(f"✅ Data split complete!")
print(f"\nTraining set:")
print(f"  X_train: {X_train.shape}")
print(f"  y_train: {y_train.shape}")
print(f"  Distribution: {np.bincount(y_train)}")

print(f"\nTesting set:")
print(f"  X_test: {X_test.shape}")
print(f"  y_test: {y_test.shape}")
print(f"  Distribution: {np.bincount(y_test)}")


✅ Data split complete!

Training set:
  X_train: (1591, 2011)
  y_train: (1591,)
  Distribution: [739 852]

Testing set:
  X_test: (398, 2011)
  y_test: (398,)
  Distribution: [185 213]


### Feature Scaling
Standardizing features to have mean=0 and std=1.

In [40]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"✅ Features scaled!")
print(f"\nScaling statistics:")
print(f"  Mean: {X_train_scaled.mean():.6f} (should be ~0)")
print(f"  Std: {X_train_scaled.std():.6f} (should be ~1)")

✅ Features scaled!

Scaling statistics:
  Mean: 0.000000 (should be ~0)
  Std: 1.000000 (should be ~1)


### Saving Preprocessed Data
Saving all processed data and models for later use.

In [41]:
# Save numpy arrays
np.save('../data/processed/X_train_scaled.npy', X_train_scaled)
np.save('../data/processed/X_test_scaled.npy', X_test_scaled)
np.save('../data/processed/y_train.npy', y_train)
np.save('../data/processed/y_test.npy', y_test)

print("✅ Saved numpy arrays:")
print("   - X_train.npy")
print("   - X_test.npy")
print("   - y_train.npy")
print("   - y_test.npy")

# Save models and metadata
# 1. Scaler ko models folder mein bhejein
with open('../models/scaler.pkl', 'wb') as f:
    pickle.dump(scaler, f)
print("\n✅ Saved scaler.pkl to models/")

# 2. TF-IDF Vectorizer ko models folder mein bhejein
with open('../models/tfidf_vectorizer.pkl', 'wb') as f:
    pickle.dump(tfidf, f)
print("✅ Saved tfidf_vectorizer.pkl to models/")

# 3. Feature names ko models folder mein bhejein
with open('../models/feature_names.pkl', 'wb') as f:
    pickle.dump(feature_df.columns.tolist(), f)
print("✅ Saved feature_names.pkl to models/")

# 4. Processed CSV ko data folder mein bhejein
df.to_csv('../data/processed_data.csv', index=False)
print("✅ Saved processed_data.csv to data/")

✅ Saved numpy arrays:
   - X_train.npy
   - X_test.npy
   - y_train.npy
   - y_test.npy

✅ Saved scaler.pkl to models/
✅ Saved tfidf_vectorizer.pkl to models/
✅ Saved feature_names.pkl to models/
✅ Saved processed_data.csv to data/


### Preprocessing Summary

✅ **Completed Steps:**
1. Date feature extraction
2. Missing value handling
3. News text combination
4. Text feature engineering
5. TF-IDF vectorization (500 features)
6. Feature combination (511 total features)
7. Train-test split (80-20)
8. Feature scaling
9. Saved all outputs

🚀 **Next Steps:**
- Model training
- Model evaluation
- Streamlit app development